# 01 — Export reduced H5 from PIConGPU openPMD

This notebook creates the reduced `.h5` file used by the paper-figure notebook. The `.h5` file is generated locally on scratch and should not be committed to Git.

In [ ]:
%matplotlib inline

# Local Git package import without installing anything
from pathlib import Path
import sys

def find_repo_root(start=None, marker=("src", "nanoplasma_analysis")):
    """Find repo root by walking upward until src/nanoplasma_analysis exists."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        candidate = p / marker[0] / marker[1]
        if candidate.exists() and candidate.is_dir():
            return p
    raise FileNotFoundError(
        f"Could not find repo root from {start}. Expected src/nanoplasma_analysis."
    )

REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nanoplasma_analysis import NanoPlasmaRun, extract_step_from_filename

print("Using repo:", REPO)
print("Using source:", SRC)

import numpy as np
import h5py
import matplotlib.pyplot as plt
import scipy.constants as sc
from pathlib import Path

## Select raw simulation and output H5

In [ ]:
# EDIT THIS: simOutput folder, not openPMD itself
SIM_OUTPUT = Path("/e/scratch/jureap18/medina2/004_V1_LowDensity/simOutput")

# Output reduced H5. Keep it outside Git. This example writes next to the run folder.
OUT_H5 = SIM_OUTPUT.parent / f"{SIM_OUTPUT.parent.name}_reduced.h5"

LASER_PEAK_STEP = 89603

print("SIM_OUTPUT:", SIM_OUTPUT)
print("openPMD exists:", (SIM_OUTPUT / "openPMD").exists())
print("OUT_H5:", OUT_H5)

In [ ]:
bp5_files = sorted((SIM_OUTPUT / "openPMD").glob("*.bp5"))
print("Number of bp5 files:", len(bp5_files))
print("First:", bp5_files[0] if bp5_files else None)
print("Last :", bp5_files[-1] if bp5_files else None)

## Run the export

This reads the raw `.bp5` output and writes a compact reduced-H5 archive with timeseries, spectra, radial diagnostics, VMI-like `px/py` histograms, and slices for the paper storyboard.

In [ ]:
run = NanoPlasmaRun(
    path=str(SIM_OUTPUT),
    laser_peak_at_target=LASER_PEAK_STEP,
)

out = run.export_reduced_h5(
    out_h5=str(OUT_H5),

    ion_species="He_i",
    e_species="He_e",
    Zmax=2,

    # Radial profiles
    r_max_nm=500.0,
    n_r=2000,

    # Angular and energy spectra
    mu_bins=720,
    energy_binning="log",
    E_min_eV=1e-3,
    E_max_eV=2000.0,
    n_E=1200,

    # VMI-like px/py histograms
    store_pxy=True,
    p_bins=256,

    # Required for the paper Figure 1 storyboard
    store_slices=True,
    slice_every=1,
    slice_y_nm=300.0,
    electron_density_field="He_e_all_density",
    ion_charge_density_field="He_i_all_density",

    overwrite=True,
    verbose=True,
)

print("Wrote:", out)

## Sanity check the reduced H5

In [ ]:
with h5py.File(OUT_H5, "r") as h5:
    print("Top-level groups:")
    for k in h5.keys():
        print(" ", k)

    print("\nAxes:")
    if "axes" in h5:
        for k in h5["axes"].keys():
            print(" ", k, h5["axes"][k].shape)

    print("\nTimeseries:")
    if "timeseries" in h5:
        for k in h5["timeseries"].keys():
            print(" ", k, h5["timeseries"][k].shape)

    print("\nSlices exists:", "slices" in h5)
    if "slices" in h5:
        print("Slice steps:", list(h5["slices"].keys())[:5], "...")

## Minimal plot from reduced H5

In [ ]:
with h5py.File(OUT_H5, "r") as h5:
    ts = h5["timeseries"]
    print("Available timeseries:", list(ts.keys()))

    # Try common names. Adjust after looking at the printed keys.
    t_key = "time_fs" if "time_fs" in ts else list(ts.keys())[0]
    t = ts[t_key][...]

    for y_key in ["N_e", "Ne", "electron_number", "num_electrons"]:
        if y_key in ts:
            y = ts[y_key][...]
            plt.figure(figsize=(7, 4))
            plt.plot(t, y)
            plt.xlabel(t_key)
            plt.ylabel(y_key)
            plt.grid(True)
            plt.tight_layout()
            plt.show()
            break
    else:
        print("No standard electron-number key found. Check printed timeseries keys above.")